# Tp03 : Étude de cas Yelp

## Création du Dataset

### Import des librairie

In [32]:
from pandas import read_csv, merge, DataFrame
from numpy import nan, floor
from utils import create_data_path, parse_hours
from matplotlib.pyplot import figure, show
from ipywidgets import Dropdown, interact

### Variable

In [33]:
export_path: str = "./restaurants_features.csv"
days: list[str] = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
data_set_path: dict[str, str] = {
    "avis"         : create_data_path("avis.csv"),
    "categories"   : create_data_path("categories.csv"),
    "checkin"      : create_data_path("checkin.csv"),
    "conseils"     : create_data_path("conseils.csv"),
    "horaires"     : create_data_path("horaires.csv"),
    "restaurants"  : create_data_path("restaurants.csv"),
    "services"     : create_data_path("services.csv"),
    "utilisateurs" : create_data_path("utilisateurs.csv")
}

c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03


### Charger Dataset

In [34]:
df_avis         = read_csv(data_set_path["avis"])
df_categories   = read_csv(data_set_path["categories"])
df_checking     = read_csv(data_set_path["checkin"])
df_conseils     = read_csv(data_set_path["conseils"])
df_horaires     = read_csv(data_set_path["horaires"])
df_restaurants  = read_csv(data_set_path["restaurants"])
df_services     = read_csv(data_set_path["services"])
df_utilisateurs = read_csv(data_set_path["utilisateurs"])

In [35]:
data = DataFrame(df_restaurants)
data = data.drop(["zone", "ferme"], axis = 1)

In [36]:
review_count = (df_avis
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_count_total"))

In [37]:
positive_reviews = (df_avis[df_avis["etoiles"] >= 4]
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_positive"))

In [38]:
review = merge(review_count, 
               positive_reviews, 
               left_on= "restaurant_id", 
               right_on="restaurant_id", 
               how="inner")

In [39]:
data = data.merge(review, 
             left_on  = "restaurant_id", 
             right_on = "restaurant_id", 
             how      = "inner")

In [40]:
data["review_count_total"] = (data["review_count_total"]
    .fillna(nan)
    .astype("int64"))

In [41]:
data["review_positive"] = (data["review_positive"]
    .fillna(nan)
    .astype("int64"))

In [42]:
data["positive_ratio"] = floor((data["review_positive"] / data["review_count_total"]) * 100) / 100

In [43]:
checkins_total = (df_checking.groupby("restaurant_id")
    .size()
    .reset_index(name="checkins_total"))

In [44]:
data = data.merge(checkins_total, 
             left_on = "restaurant_id", 
             right_on = "restaurant_id", 
             how = "inner")

In [45]:
data["checkins_total"] = data["checkins_total"].fillna(nan).astype("int64")

In [46]:
name_counts = data["nom"].value_counts()

In [47]:
data["is_chain"] = data["nom"].map(lambda x: name_counts[x] >= 3)

In [48]:
prix_moyen = (df_services
    .groupby("restaurant_id")["prix"]
    .mean()
    .reset_index(name="prix_moyen"))

In [49]:
data = data.merge(prix_moyen, 
                  left_on = "restaurant_id", 
                  right_on = "restaurant_id", 
                  how = "left")

In [50]:
df_elite = df_utilisateurs[df_utilisateurs["elite"].notna()]

In [51]:
df_elite = df_utilisateurs[df_utilisateurs["elite"] != "[]"]

In [52]:
elite_id = df_elite[["utilisateur_id"]]

In [53]:
elite_reviews = df_avis.merge(elite_id,
    on="utilisateur_id",
    how="inner")


In [54]:
elite_users_count = (elite_reviews
                     .groupby("restaurant_id")["utilisateur_id"]
                     .nunique()
                     .reset_index(name="elite_users_count"))

In [55]:
data = data.merge(elite_users_count,
    on="restaurant_id",
    how="left")

In [56]:
for day in days:
    df_horaires[day] = df_horaires[day].apply(parse_hours)

In [57]:
df_horaires["avg_open_hours"] = df_horaires[days].mean(axis=1)

In [58]:
df_horaires["avg_open_hours"] = floor(df_horaires["avg_open_hours"] * 100) / 100

In [59]:
data = data.merge(df_horaires[["restaurant_id", "avg_open_hours"]],
    on="restaurant_id",
    how="left")

In [60]:
data.to_csv(export_path)

In [61]:
numeric_columns = data.select_dtypes('number').columns

x_dropdown = Dropdown(options=numeric_columns, description='X')
y_dropdown = Dropdown(options=numeric_columns, description='Y')
z_dropdown = Dropdown(options=numeric_columns, description='Z')

city_dropdown = Dropdown(
    options=['(All)'] + sorted(data['ville'].unique()),
    description='Ville'
)

@interact(x=x_dropdown, y=y_dropdown, z=z_dropdown, city=city_dropdown)
def update_plot(x, y, z, city):
    ville = data if city == '(All)' else data[data['ville'] == city]

    fig = figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(ville[x], ville[y], ville[z], s=40, alpha=0.8)

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_zlabel(z)
    ax.set_title(f"Scatter 3D : {x} vs {y} vs {z}\nVille : {city}")
    
    show()

interactive(children=(Dropdown(description='X', options=('moyenne_etoiles', 'review_count_total', 'review_posi…